In [1]:
print(123)

123


In [2]:
import sys
!{sys.executable} -m pip install python-dotenv

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
import sys
!{sys.executable} -m pip install requests minsearch openai jupyter

In [5]:
from openai import OpenAI

In [6]:
openai_client = OpenAI()

In [7]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [8]:
llm('hey whats up')

'Hey! Not much — I’m here and ready to help. What’s up with you?'

In [9]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Absolutely — if the course is still open, you can usually join now.

A quick note: if it already started, you may still be able to enroll and catch up, depending on the program’s policy. If you want, I can help you check whether:
- enrollment is still open,
- there’s a late start option,
- or you need to contact the instructor/admin.

If you send me the course name or platform, I can help you figure it out.


In [10]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [11]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [12]:
answer = llm(prompt)
print(answer)

Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still open.


In [13]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [14]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [15]:
courses_raw

[{'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 471},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 253},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 404},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 118}]

In [16]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1380

In [17]:
documents[0]

{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

In [18]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [19]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '193612db63',
  'course': 'llm-zoomcamp',
  'section': 'Module 3: Orchestration',
  'question': "Why do we need orchestration / Kestra — can't I just run the code in a notebook?",
  'answer': "Notebooks are grea

In [20]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 "Why do we need orchestration / Kestra — can't I just run the code in a notebook?",
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?']

In [21]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [22]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [23]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [24]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [25]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

Module 3: Orchestration
Q: Why do we need orchestration / Kestra — can't I just run the code in a notebook?
A: Notebooks are great for learning and experimenting, but real AI workflows need more than a script that runs once: scheduling, retries, monitoring, secret management, and reliably chaining tasks together. That's what an orchest